# Credential Usage Analysis

Which seeded creds are getting hit, how fast after deploy, and from where.

Data source: cowrie events from ingest-elk-01:9200 (index: `cowrie-*`)

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from elasticsearch import Elasticsearch
import json

es = Elasticsearch(['http://ingest-elk-01:9200'])

SEEDED_CREDS = {
    'deploy/NullDeploy#2026': 'github-leaked',
    'postgres/FintechSecure2026!': 'github-leaked',
    'admin/admin': 'common-default',
    'root/root': 'common-default',
    'root/123456': 'common-default',
}

SENSORS = ['sensor-us-01', 'sensor-apac-01', 'sensor-eu-01']

In [2]:
def query_login_attempts(days_back=30, size=10000):
    """Pull login attempts from ES. Returns raw hits."""
    q = {
        'query': {
            'bool': {
                'must': [
                    {'term': {'eventid.keyword': 'cowrie.login.success'}},
                    {'range': {'@timestamp': {'gte': f'now-{days_back}d'}}}
                ]
            }
        },
        'size': size,
        '_source': ['@timestamp', 'src_ip', 'username', 'password', 'sensor', 'session']
    }
    resp = es.search(index='cowrie-*', body=q)
    return [h['_source'] for h in resp['hits']['hits']]

In [3]:
raw = query_login_attempts(days_back=14)
df = pd.DataFrame(raw)
df['@timestamp'] = pd.to_datetime(df['@timestamp'])
df['cred_pair'] = df['username'] + '/' + df['password']
df['is_seeded'] = df['cred_pair'].isin(SEEDED_CREDS.keys())
print(f'Loaded {len(df):,} login events')

Loaded 4,217 login events


In [ ]:
# old approach - dont use, the agg query was wrong
# resp = es.search(index='cowrie-*', body={
#     'size': 0,
#     'aggs': {
#         'by_cred': {
#             'terms': {'script': "doc['username.keyword'].value + '/' + doc['password.keyword'].value", 'size': 50},
#             'aggs': {'first_seen': {'min': {'field': '@timestamp'}}}
#         }
#     }
# })
# # this was super slow and kept timing out on large indices

## Seeded vs common creds breakdown

In [7]:
df['is_seeded'].value_counts()

is_seeded
False    3894
True      323
Name: count, dtype: int64

In [8]:
cred_stats = df.groupby('cred_pair').agg(
    count=('cred_pair', 'size'),
    first_attempt=('@timestamp', 'min'),
    last_attempt=('@timestamp', 'max'),
    unique_ips=('src_ip', 'nunique')
).reset_index().sort_values('count', ascending=False).head(10)

cred_stats

                          cred_pair  count first_attempt          last_attempt  unique_ips
0              root/root             1847   2026-03-14 02:11:33  2026-03-27 23:44:18     412
1              admin/admin            891   2026-03-14 02:14:07  2026-03-27 22:58:01     287
2              root/123456            614   2026-03-14 03:01:44  2026-03-27 21:30:55     198
3   deploy/NullDeploy#2026           201   2026-03-16 14:22:08  2026-03-27 19:11:43      34
4              root/password          188   2026-03-14 04:33:21  2026-03-27 18:45:12     143
5   postgres/FintechSecure2026!       122   2026-03-17 08:55:31  2026-03-27 16:02:44      19
6              admin/123456            109   2026-03-14 05:12:09  2026-03-27 14:23:55      88
7              root/toor               78   2026-03-14 06:44:32  2026-03-26 11:08:22      61
8              ubuntu/ubuntu            67   2026-03-15 01:23:11  2026-03-27 09:44:08      44
9              test/test                55   2026-03-14 09:18:45  2026-

interesting - the seeded `deploy` cred showed up ~2 days after we pushed the fake repo. `postgres` took 3 days. both have way fewer unique IPs than the bruteforce defaults which makes sense -- these are targeted, not spray-and-pray

In [9]:
# per sensor - eu-01 is offline so wont show up
df.groupby('sensor').size()

sensor
sensor-us-01      2301
sensor-apac-01    1916
Name: count, dtype: int64

In [10]:
df.groupby(['sensor', 'is_seeded']).size()

sensor          is_seeded
sensor-apac-01  False        1778
                True          138
sensor-us-01    False        2116
                True          185
Name: count, dtype: int64

## Time-to-first-use for seeded creds

github repo was pushed 2026-03-14 ~12:00 UTC

In [11]:
repo_push_time = pd.Timestamp('2026-03-14 12:00:00', tz='UTC')

seeded_only = df[df['is_seeded']]
for cred in ['deploy/NullDeploy#2026', 'postgres/FintechSecure2026!']:
    subset = seeded_only[seeded_only['cred_pair'] == cred]
    if len(subset) > 0:
        first = subset['@timestamp'].min()
        delta = first - repo_push_time
        print(f'{cred}:')
        print(f'  first seen: {first}')
        print(f'  time to first use: {delta}')
        print()

deploy/NullDeploy#2026:
  first seen: 2026-03-16 14:22:08
  time to first use: 2 days, 2:22:08

postgres/FintechSecure2026!:
  first seen: 2026-03-17 08:55:31
  time to first use: 2 days, 20:55:31



In [12]:
# who used seeded creds first? IP overlap?
seeded_by_ip = seeded_only.groupby(['src_ip', 'cred_pair']).agg(
    first_seen=('@timestamp', 'min'),
    sessions=('session', 'nunique')
).reset_index().sort_values('first_seen').head()

seeded_by_ip

                        src_ip          cred_pair           first_seen  sessions
0              45.148.10.XXX   deploy/NullDeploy#2026   2026-03-16 14:22:08       8
1              185.220.101.XX  deploy/NullDeploy#2026   2026-03-16 19:45:33       3
2              194.26.29.XXX   postgres/FintechSecure..  2026-03-17 08:55:31       5
3              89.248.165.XX   deploy/NullDeploy#2026   2026-03-18 03:12:44       2
4              45.148.10.XXX   postgres/FintechSecure..  2026-03-18 11:30:22       4

45.148.10.XXX hit both creds across sensors. need to check if same actor or if that IP block is a known botnet. cross-ref with session commands.

---

## todo